# Paper Reproduction Notebook

This notebook is the publication-ready reproduction path for `refs/paperjrss/paper.tex`. It recomputes the paper artifacts from the repository data and scripts: backend symbolic identification and AST translation, synthetic HCM benchmarks, STAR baselines, STAR ExactBIC HCM estimates, paper-ready figures, and the final PDF.

The heavy numerical routines are not duplicated here. Each cell calls the source scripts in `examples/new` and `examples/STAR`, then reloads the generated artifacts so the numbers shown in the notebook match the paper.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find repository root containing pyproject.toml")

REPO = find_repo_root(Path.cwd().resolve())

STAR = REPO / "examples" / "STAR"
PAPER = REPO / "refs" / "paperjrss"
FIG = PAPER / "Fig"
RESULTS = STAR / "results"

def run(cmd: list[str], cwd: Path = REPO) -> None:
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

print(REPO)

## 1. Backend identification and AST representation

This cell recomputes the do-calculus example used in the method section of the paper. The example is `ID_ex6_collapse` from `examples/new/collapsed_cases.py`. The displayed formula is the symbolic estimand returned by pyAgrum after HCM collapse, and the printed block is the adapted AST tree used by the evaluator.

In [ ]:
import re
import sys
from IPython.display import Math, display

EXAMPLES_NEW = REPO / "examples" / "new"
if str(EXAMPLES_NEW) not in sys.path:
    sys.path.insert(0, str(EXAMPLES_NEW))

import hierarchicalcausalmodels.do_calculus as dc
from collapsed_cases import COLLAPSED_DO_CALCULUS_CASES, build_cgm_for_case

def latex_for_notebook(value: str) -> str:
    return re.sub(r"(?<=[A-Za-z0-9])_(?=[A-Za-z0-9])", r"\\_", value)

def canonical_id_ex6_ast(value: str) -> str:
    value = value.replace(
        "| | | P(Y|Qa,Qz_a,W)\n| | | P(W|Qa)",
        "| | | P(W|Qa)\n| | | P(Y|Qa,Qz_a,W)",
    )
    return value.replace(
        "| | | | joint P(Qa)\n| | | | P(Qz_a|Qa,W)",
        "| | | | P(Qz_a|Qa,W)\n| | | | joint P(Qa)",
    )

case = next(row for row in COLLAPSED_DO_CALCULUS_CASES if row[0] == "ID_ex6_collapse")
cgm, unobserved, outcome_node, intervention_node, _expected_identifiable = build_cgm_for_case(case)
cgm.unobserved_variables = unobserved
result = dc.identify_effect(cgm, Y=outcome_node, X=intervention_node, unobserved=unobserved)

if not result.identifiable or result.formula_latex is None or result.ast is None:
    raise RuntimeError(result.error or result.explanation or "ID_ex6_collapse was not identified")

print(f"Case: {case[0]}")
print(f"Outcome (Y): {outcome_node}, Intervention (X): {intervention_node}")
display(Math(latex_for_notebook(result.formula_latex)))
print("AST:", canonical_id_ex6_ast(str(result.ast)))

## 2. Synthetic validation results

The paper reports canonical motifs and benchmark scenarios. The historical full benchmark notebook is `examples/new/benchmarks.ipynb`; the script entry point is `examples/new/benchmarks.py`. Run the script if you want to regenerate the synthetic validation artifacts from scratch.

### Canonical motif parallel estimation table

The convergence plot reports ATE recovery, so the paper table for the three canonical motifs focuses on the independent estimation work exposed by the AST. CPU parallelism is expressed as $P=4$ workers. GPU parallelism is measured as a single batched CUDA workload: on the local RTX 4060 Laptop GPU, the benchmark searches for the largest successful batch of independent HCM unit tasks.

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_limit_path = PAPER / "gpu_hcm_batched_benchmark_limit.json"
    if not gpu_limit_path.exists():
        run(["uv", "run", "python", "refs/paperjrss/benchmark_gpu_batched_hcm.py"])
        gpu_limit_path = PAPER / "gpu_hcm_batched_benchmark.json"
    gpu_benchmark = json.loads(gpu_limit_path.read_text())
    best_gpu_batch = gpu_benchmark["best_gpu_batch"]
    max_gpu_tasks = int(best_gpu_batch["tasks"])
else:
    gpu_benchmark = {"device": "CUDA unavailable", "best_gpu_batch": None}
    max_gpu_tasks = 0

canonical_parallel_table = pd.DataFrame([
    {
        "Motif": "Confounder",
        "Independent tasks": "n response tasks",
        "CPU work, P=4": "ceil(n / 4) c",
        "GPU batch ceiling on local config": f"up to {max_gpu_tasks / 1_000_000:.1f}M units" if max_gpu_tasks else "CUDA unavailable",
    },
    {
        "Motif": "Confounder and interference",
        "Independent tasks": "2n response/mediator tasks",
        "CPU work, P=4": "ceil(2n / 4) c",
        "GPU batch ceiling on local config": f"up to {max_gpu_tasks / 2 / 1_000_000:.1f}M units" if max_gpu_tasks else "CUDA unavailable",
    },
    {
        "Motif": "Instrument",
        "Independent tasks": "3n instrument/weight tasks",
        "CPU work, P=4": "ceil(3n / 4) c",
        "GPU batch ceiling on local config": f"up to {max_gpu_tasks / 3 / 1_000_000:.1f}M units" if max_gpu_tasks else "CUDA unavailable",
    },
])

canonical_parallel_table

In [ ]:
if gpu_benchmark["best_gpu_batch"] is not None:
    best_gpu_batch = gpu_benchmark["best_gpu_batch"]
    pd.DataFrame([
        {
            "device": gpu_benchmark["device"],
            "torch": gpu_benchmark["torch"],
            "n_subunits": gpu_benchmark["n_subunits"],
            "largest_successful_tasks": best_gpu_batch["tasks"],
            "seconds": best_gpu_batch["gpu_seconds"],
            "peak_memory_gb": best_gpu_batch["gpu_peak_memory_gb"],
            "tasks_per_second": best_gpu_batch["tasks_per_second"],
        }
    ]).round({"seconds": 2, "peak_memory_gb": 2, "tasks_per_second": 0})
else:
    pd.DataFrame([{"device": gpu_benchmark["device"]}])

In [ ]:
# Synthetic benchmark regeneration. This can be slower than loading the stored results.
run(["uv", "run", "python", "examples/new/benchmarks.py"])

## 3. STAR baselines and graph-family benchmark

This reproduces the OLS/IV baselines and HCM graph-family comparison used to motivate why the flat analysis and the hierarchical causal estimand are not the same target.

In [ ]:
run(["uv", "run", "python", "examples/STAR/star_baseline_and_hcm_benchmark.py", "--outcome", "math"])
bench = json.loads((RESULTS / "star_baseline_math_benchmark.json").read_text())
ols_rows = []
for name, row in bench["econometric_baselines"]["ols"].items():
    ols_rows.append({"model": name, "coef_small": row.get("coef_small"), "se_small": row.get("se_small"), "r2": row.get("r2"), "seconds": row.get("elapsed_seconds")})
pd.DataFrame(ols_rows)

In [ ]:
iv = bench["econometric_baselines"]["iv"]["iv_2sls"]
pd.DataFrame([iv])[["coef_classsize", "se_classsize", "p_classsize", "elapsed_seconds"]]

## 4. STAR ExactBIC HCM artifacts

The mathematics table in the paper is read from `star_lingam_bic_gaussian_artifacts/index.json`, selecting `graph=ExactBIC` and `outcome_subunit=M`. The normalized-factor diagnostic is a separate Reading-mechanism artifact (`outcome_subunit=Y`) and is displayed separately to avoid mixing targets.

In [ ]:
# Regenerate the HCM v2 run and the gaussian artifact index used by the paper.
run(["uv", "run", "python", "examples/STAR/star_hcm_v2_teacher_student.py"])
run(["uv", "run", "python", "examples/STAR/star_export_lingam_bic_gaussian_artifacts.py"])

index_path = RESULTS / "star_lingam_bic_gaussian_artifacts" / "index.json"
artifact_index = json.loads(index_path.read_text())
rows = []
for row in artifact_index["runs"]:
    if row["graph"] == "ExactBIC":
        rows.append({
            "graph": row["graph"],
            "outcome_subunit": row["outcome_subunit"],
            "E_do_1": row["E_do_1"],
            "E_do_0": row["E_do_0"],
            "ATE": row["ATE"],
            "families": row["distribution_families"],
        })
pd.DataFrame(rows)

## 5. Parallel runtime, speedups, and convergence diagnostics

This section reproduces the new computational diagnostics used in the paper. The STAR HCM timings are CPU timings from the stored speed-test JSON: sequential evaluation, four CPU threads, and four CPU processes. The convergence diagnostic is recomputed below and reports the runtime at each sample size. The same AST decomposition is GPU-compatible because unit-level evaluations, factor evaluations, intervention values, and Monte Carlo samples can be batched in a tensor backend, although the reported STAR timings here are CPU runs.

In [ ]:
speed_path = RESULTS / "ate_10_40_parallel_speed_test.json"
baseline_parallel_path = RESULTS / "parallel_benchmark_results.json"
speed = json.loads(speed_path.read_text())
baseline_parallel = json.loads(baseline_parallel_path.read_text())

modes = ["seq", "threads4", "proc4"]
mode_labels = {"seq": "Sequential", "threads4": "4 threads", "proc4": "4 processes"}

detailed_rows = []
for graph, payload in speed["graphs"].items():
    seq_time = payload["modes"]["seq"]["elapsed_seconds_total_do1_do0"]
    for mode in modes:
        vals = payload["modes"][mode]
        detailed_rows.append({
            "workload": f"{graph} HCM evaluation",
            "mode": mode_labels[mode],
            "n_jobs": vals["n_jobs"],
            "backend": vals["parallel_backend"],
            "ATE": vals["ATE"],
            "seconds": vals["elapsed_seconds_total_do1_do0"],
            "speedup_vs_seq": seq_time / vals["elapsed_seconds_total_do1_do0"],
        })

detailed_runtime = pd.DataFrame(detailed_rows)

def workload_summary(name: str, values: dict[str, float]) -> dict[str, float | str]:
    sequential = values["seq"]
    threads = values["threads4"]
    processes = values["proc4"]
    return {
        "Workload": name,
        "Sequential": sequential,
        "4 threads": threads,
        "4 processes": processes,
        "Best speedup": sequential / min(sequential, threads, processes),
    }

paper_runtime_table = pd.DataFrame([
    workload_summary(
        "DirectLiNGAM HCM evaluation",
        {mode: speed["graphs"]["DirectLiNGAM"]["modes"][mode]["elapsed_seconds_total_do1_do0"] for mode in modes},
    ),
    workload_summary(
        "ExactBIC HCM evaluation",
        {mode: speed["graphs"]["ExactBIC"]["modes"][mode]["elapsed_seconds_total_do1_do0"] for mode in modes},
    ),
    workload_summary(
        "OLS baseline batch",
        {mode: baseline_parallel[mode]["ols_seconds"] for mode in modes},
    ),
    workload_summary(
        "IV baseline batch",
        {mode: baseline_parallel[mode]["iv_seconds"] for mode in modes},
    ),
])

paper_runtime_table.round({"Sequential": 2, "4 threads": 2, "4 processes": 2, "Best speedup": 2})

In [ ]:
detailed_runtime[["workload", "mode", "n_jobs", "backend", "ATE", "seconds", "speedup_vs_seq"]].round({
    "ATE": 4,
    "seconds": 3,
    "speedup_vs_seq": 3,
})

### Convergence plot with runtime labels

The convergence plot is not only a visual replacement for a numerical table. It also reports the runtime at each sample size, showing how the same independent unit-level computations scale as $n\times m$ grows.

In [ ]:
import importlib
from IPython.display import Image, display

if str(PAPER) not in sys.path:
    sys.path.insert(0, str(PAPER))

import make_performance_figures as performance_figures
performance_figures = importlib.reload(performance_figures)

convergence = performance_figures.convergence_points()
convergence_table = pd.DataFrame({
    "total_sample_n_times_m": convergence["total_sample"].astype(int),
    "flat_pooled_rmse": convergence["flat_rmse"],
    "hcm_within_unit_rmse": convergence["hcm_rmse"],
    "reference_1_over_sqrt_nm": convergence["reference"],
    "runtime_seconds": convergence["runtime_seconds"],
})

performance_figures.style()
performance_figures.plot_convergence(convergence)

display(convergence_table.round({
    "flat_pooled_rmse": 4,
    "hcm_within_unit_rmse": 4,
    "reference_1_over_sqrt_nm": 4,
    "runtime_seconds": 3,
}))
display(Image(filename=str(FIG / "synthetic_hcm_convergence_rmse.png")))

## 6. Paper-ready figures and diagnostics

This regenerates the figures stored in `refs/paperjrss/Fig`, including the ExactBIC nominal/effective graph, the transformed Q-variable graph, and the convergence diagnostic with runtime labels. Sequential versus parallel runtimes are reported as a table in the paper.

In [ ]:
run(["uv", "run", "python", "star_make_paper_ready_figures.py"], cwd=STAR)
run(["uv", "run", "python", "make_performance_figures.py"], cwd=PAPER)
figures = sorted(p.name for p in FIG.glob("*.pdf"))
pd.DataFrame({"paper_figure_pdf": figures})

## 7. Compile the paper

The final cell compiles the OUP-style paper PDF from the regenerated artifacts.

In [ ]:
run(["latexmk", "-pdf", "-interaction=nonstopmode", "-halt-on-error", "paper.tex"], cwd=PAPER)
print(PAPER / "paper.pdf")